In [ ]:
from datasets import *
from network import *
from loss import *
from training import *

In [ ]:
H5_PATH = "training.h5"
BATCH_SIZE = 16
dataset = LArTPCSequenceDataset(H5_PATH)

In [ ]:
from torch.utils.data import Subset
dataset = Subset(dataset, range(3000))

from torch.utils.data import random_split

val_frac = 0.1
num_events = len(dataset)
num_val = int(num_events * val_frac)
num_train = num_events - num_val

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(dataset, [num_train, num_val], generator=generator)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=False, num_workers=0)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model_kwargs = dict(
    input_dim=3,          # xx, zz, vv
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    ff_dim=256,
    num_classes=None,     # OC doesn't need this
    dropout=0.1
)
# Set checkpoint_path to None if starting new
model = load_or_initialize_model(ObjectCondensationNet, model_kwargs, checkpoint_path="model_epoch_3.pt", device=device)
#model = load_or_initialize_model(ObjectCondensationNet, model_kwargs, checkpoint_path=None, device=device)

In [ ]:
criterion = ObjectCondensationLoss()

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

In [ ]:
scaler = GradScaler()

In [ ]:
num_epochs = 1

## Training Loop

In [ ]:
best_val_loss = float("inf")

for epoch in range(1, num_epochs + 1):
    print(f"\n===== Epoch {epoch}/{num_epochs} =====")

    train_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        scaler=scaler,
        max_grad_norm=1.0
    )

    save_checkpoint(model, optimizer, num_epochs, f"model_epoch_{epoch}.pt")

    val_loss = validate(
        model=model,
        val_loader=val_loader,
        criterion=criterion,
        device=device
    )

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    # ---------------------------------
    # Save best model
    # ---------------------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, num_epochs, f"best_model.pt")
        print("  → Saved best model")

## Inference and Metrics

In [ ]:
from clustering import *
from metrics import *

@torch.no_grad()
def evaluate_oc(model, loader, device, beta_threshold=0.1):
    model.eval()

    metrics_accum = {
        "ARI": [],
        "purity": [],
        "eff": [],
        "f1": [],
        "bg_rej": [],
    }

    for batch in loader:
        hits = batch["hits"].to(device)                # (B, N, F)
        slice_labels = batch["slice_labels"]           # (B, N)
        cp_labels = batch["cp_labels"]                 # optional

        pred = model(hits)
        beta = pred["beta"]   # (B, N, 1)
        embed = pred["embed"] # (B, N, D)

        B = hits.shape[0]

        for i in range(B):
            beta_i = beta[i]                 # (N,1)
            embed_i = embed[i]               # (N,D)
            truth_i = slice_labels[i].numpy()

            cluster_ids, cp_indices = oc_cluster(beta_i, embed_i, beta_threshold)

            m = compute_instance_metrics(cluster_ids, truth_i)

            metrics_accum["ARI"].append(m["ARI"])
            metrics_accum["purity"].append(m["purity_mean"])
            metrics_accum["eff"].append(m["eff_mean"])
            metrics_accum["f1"].append(m["f1_mean"])
            metrics_accum["bg_rej"].append(m["bg_rejection"])

    # Average over all events
    return {k: float(np.mean(v)) for k, v in metrics_accum.items()}

In [ ]:
evaluate_oc(model, val_loader, device, beta_threshold=0.1)

## Event display

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

def plot_event_display(batch, cluster_assignments, event_index=0, figsize=(6,6), cmap="tab20"):
    """
    Produce an event display for a single event in a batch.

    Args:
        batch: dict from the DataLoader containing:
            - batch["hits"]:  (B, N, F) where features include x, z (assumed first two dims)
            - batch["mask"]:  (B, N)
        cluster_assignments: torch.LongTensor or np.ndarray of shape (B, N),
                             containing the cluster ID for each hit.
        event_index: which event inside the batch to display
        figsize: matplotlib figure size
        cmap: matplotlib colormap name
    """

    hits = batch["hits"][event_index]      # (N, F)
    mask = batch["mask"][event_index]      # (N)
    clusters = cluster_assignments[event_index]

    # Move to CPU + numpy
    if isinstance(hits, torch.Tensor):
        hits = hits.detach().cpu().numpy()
    if isinstance(mask, torch.Tensor):
        mask = mask.detach().cpu().numpy()
    if isinstance(clusters, torch.Tensor):
        clusters = clusters.detach().cpu().numpy()

    # Apply mask (remove padded hits)
    hits = hits[mask]
    clusters = clusters[mask]

    x = hits[:, 0]
    z = hits[:, 1]

    plt.figure(figsize=figsize)
    scatter = plt.scatter(x, z, c=clusters, cmap=cmap, s=6, alpha=0.9)

    plt.xlabel("x")
    plt.ylabel("z")
    plt.title(f"Event Display — event {event_index} in batch")
    plt.gca().set_aspect("equal", adjustable="box")
    plt.colorbar(scatter, label="Cluster ID")
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
batch = next(iter(val_loader))
batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(batch["hits"])
    beta  = outputs["beta"]      # (B, N, 1)
    embed = outputs["embed"]     # (B, N, D)

cluster_assignments = compute_cluster_assignments(
    beta, embed, batch["slice_labels"], batch["mask"]
)

# Visualize event 0
plot_event_display(batch, cluster_assignments, event_index=0)